# Exploration des données brutes Olist

**Objectif :** comprendre les données **avant** toute transformation, pour justifier les choix du pipeline dbt.

Chaque section part d'une **observation** dans les CSV bruts et se termine par le **choix** qu'elle justifie.
Les choix sont résumés dans `docs/choix_de_modelisation.md`.

> On lit directement les fichiers de `ctc_olist/data/raw/`, sans passer par dbt.
> Pour les obtenir : `./scripts/download_data.sh`

## Préparation

DuckDB peut lire un CSV comme une table. La petite fonction `sql()` lance une requête et affiche le résultat en tableau.

In [ ]:
import duckdb

# Dossier des CSV bruts (le notebook est dans notebooks/)
RAW = "../ctc_olist/data/raw"

def sql(query):
    """Lance une requête SQL et renvoie un tableau pandas."""
    return duckdb.sql(query).df()

def csv(name):
    """Chemin d'un fichier brut, à utiliser dans une requête."""
    return f"'{RAW}/{name}.csv'"

## 1. Vue d'ensemble des fichiers

**Question :** combien de fichiers, de lignes et de colonnes ? Quels types DuckDB devine-t-il ?

In [ ]:
files = [
    "olist_orders_dataset",
    "olist_customers_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation",
    "olist_geolocation_dataset",
]

# Nombre de lignes et de colonnes de chaque fichier
import pandas as pd

rows = []
for f in files:
    n_rows = sql(f"select count(*) as n from {csv(f)}")["n"][0]
    n_cols = len(sql(f"select * from {csv(f)} limit 0").columns)
    rows.append({"file": f, "rows": n_rows, "columns": n_cols})

pd.DataFrame(rows)

**À retenir :** 9 fichiers. La géolocalisation est le plus gros, mais elle ne sert pas à notre sujet (la satisfaction client) : on ne la déclare pas comme source.

### Les types devinés par DuckDB

Exemple avec les commandes et les clients.

In [ ]:
sql(f"describe select * from {csv('olist_orders_dataset')}")

In [ ]:
sql(f"describe select * from {csv('olist_customers_dataset')}")

In [ ]:
# Des codes postaux commencent par 0 : en nombre, ce zéro disparaîtrait
sql(f"""
    select customer_zip_code_prefix
    from {csv('olist_customers_dataset')}
    where customer_zip_code_prefix like '0%'
    limit 5
""")

**À retenir :**
- Les types sont **devinés** à partir du contenu du fichier : ils pourraient changer si le fichier changeait.
- Certains noms ne correspondent pas au contenu : `order_delivered_carrier_date` contient une **heure**,
  et `order_estimated_delivery_date` est lue comme date et heure alors qu'elle est toujours à minuit.
- Les codes postaux commencent parfois par **0** : il faut les garder en **texte**.

**→ Choix :** en staging, on **convertit explicitement** chaque type (`cast`), on renomme avec la convention `_at` (date et heure) / `_date` (jour), et on garde les codes postaux en texte.